# Semantic Link – Playground

**A safe what-if sandbox for Power BI semantic model measures.**

> "What would Total Sales be if I excluded returns? What if I changed the discount logic? Which other measures would be affected?"
>
> Today, answering these questions means editing the live model, refreshing, checking, and hoping you remember to revert. Playground lets you experiment with any measure's DAX — see the impact instantly — without touching the production model.

---

## The problem

Semantic model developers constantly need to answer "what if?" questions:

- *What if I change this measure's formula?*
- *How much would the number move?*
- *Which downstream KPIs would be affected?*
- *Is it safe to make this change?*

Today the only option is to edit the live model, evaluate, and revert. That's risky in production, slow in development, and impossible to do side-by-side. There's no "preview" button for DAX.

## What Playground does

Playground gives you a **safe sandbox** for measure experimentation:

1. **Explore** — Browse every measure in your model with its DAX, table, and current value.
2. **What-If** — Modify any measure's DAX in a sandbox. Playground evaluates both the original and your modified version side-by-side. The live model is never touched.
3. **Impact** — See which downstream measures reference the one you changed, and how their values shift as a result.
4. **Compare** — Visual bar charts and impact trees showing exactly what changed and by how much.

All powered by `semantic-link` + `semantic-link-labs`, running entirely in a Fabric notebook. Zero writes to the live model — every "what-if" runs via `evaluate_dax()` with your modified DAX injected inline.

## Why this matters for developer experience

| Pain today | With Playground |
|---|---|
| "I need to test a DAX change but can't risk breaking prod." | Sandbox evaluates it without touching the model. |
| "How much would this formula change move the number?" | Side-by-side comparison with exact delta. |
| "Will changing this measure break anything downstream?" | Impact cascade shows every affected KPI. |
| "I want to compare three different formula approaches." | Run multiple what-if scenarios in sequence. |


---

## Architecture at a glance

```
┌─────────────────────────┐
│  Live Semantic Model    │
│  (Power BI / Fabric)    │
└───────────┬─────────────┘
            │ sempy.fabric + sempy_labs (READ ONLY)
            ▼
┌─────────────────────────┐
│    explore_model()      │──► Measure Catalog
│  name, table, DAX, value│    (pandas DataFrame)
└───────────┬─────────────┘
            │
     ┌───────────────┼──────────────┐
     ▼               ▼              ▼
┌──────────┐ ┌──────────────┐ ┌────────────────┐
│ whatif   │ │   impact     │ │   compare      │
│ measure  │ │  analysis    │ │  scenarios     │
└────┬─────┘ └──────┬───────┘ └───────┬────────┘
     │              │                 │
     ▼              ▼                 ▼
  evaluate_dax   evaluate_dax     evaluate_dax
  (DEFINE        (DEFINE           (DEFINE
   MEASURE        MEASURE           MEASURE
   override)      override)         override)
     │              │                 │
     ▼              ▼                 ▼
  Side-by-Side   Impact Cascade   Multi-Scenario
  Bar Chart      + Delta Table    Comparison Chart
```

**Key guarantee:** Playground never calls any write API. Every evaluation uses `fabric.evaluate_dax()` with your modified DAX injected via `DEFINE MEASURE`. The live model definition is untouched throughout.


---

## Step 1 — Configuration

Edit these values for your environment. If you're using the Wide World Importers sample from the challenge getting-started guide, just change `WORKSPACE_NAME` and `DATASET_NAME`.


In [ ]:
# ---- USER CONFIG (edit this block) ----------------------------------------
WORKSPACE_NAME = "YourWorkspaceName"    # The Fabric workspace containing the model
DATASET_NAME   = "YourDatasetName"      # The semantic model name
# ---- End user config -------------------------------------------------------


## Step 2 — Install and import


In [ ]:
%pip install semantic-link-labs --quiet


In [ ]:
import sempy.fabric as fabric
import sempy_labs as labs
import pandas as pd
import json
import re
import warnings
warnings.filterwarnings("ignore")

print("sempy + sempy_labs loaded ✓")


---

## Step 3 — Explore the model

Before experimenting, let's see what's in the model: every measure with its home table, DAX expression, and current evaluated value.


In [ ]:
def _extract_measures_from_tmsl(tmsl_obj):
    """Parse measures directly from the authoritative TMSL/BIM JSON.

    This avoids the ``fabric.list_measures()`` cache that can serve stale
    DAX expressions after a recent model edit.

    Returns:
        pandas.DataFrame with columns: measure_name, table_name, expression,
        format_string, display_folder, description.
    """
    rows = []
    tables = tmsl_obj.get("model", {}).get("tables", [])
    for t in tables:
        t_name = t.get("name")
        for m in t.get("measures", []):
            expr = m.get("expression")
            if isinstance(expr, list):
                expr = "\n".join(expr)
            rows.append({
                "measure_name":   m.get("name"),
                "table_name":     t_name,
                "expression":     expr,
                "format_string":  m.get("formatString"),
                "display_folder": m.get("displayFolder"),
                "description":    m.get("description"),
            })
    return pd.DataFrame(rows)


def _evaluate_single_measure(measure_name):
    """Evaluate a single measure at total level. Returns (numeric, string)."""
    try:
        result = fabric.evaluate_measure(
            dataset=DATASET_NAME,
            workspace=WORKSPACE_NAME,
            measure=measure_name,
        )
        val = result.iloc[0, 0] if len(result) else None
        if val is not None and not pd.isna(val):
            try:
                return float(val), None
            except (TypeError, ValueError):
                return None, str(val)
        return None, None
    except Exception as e:
        return None, f"ERROR: {e}"


def explore_model(evaluate=True):
    """Load the full measure catalog from the live model.

    Reads the TMSL via ``sempy_labs``, extracts every measure, and optionally
    evaluates each one at total level via ``fabric.evaluate_measure()``.

    Args:
        evaluate: If True, evaluates every measure (slower but shows values).
                  If False, just returns definitions.

    Returns:
        pandas.DataFrame — the measure catalog. Also stored in the global
        ``MODEL_CATALOG`` variable for use by other functions.
    """
    global MODEL_CATALOG, TMSL_OBJ

    print(f"Reading model '{DATASET_NAME}' in workspace '{WORKSPACE_NAME}'...")
    tmsl = labs.get_semantic_model_bim(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
    TMSL_OBJ = tmsl if isinstance(tmsl, dict) else json.loads(tmsl)

    catalog = _extract_measures_from_tmsl(TMSL_OBJ)
    print(f"  Found {len(catalog)} measures across "
          f"{catalog['table_name'].nunique()} tables")

    if evaluate:
        print(f"  Evaluating all measures at total level...")
        values = []
        for _, row in catalog.iterrows():
            num, string = _evaluate_single_measure(row["measure_name"])
            values.append({"value_numeric": num, "value_string": string})
        vals_df = pd.DataFrame(values)
        catalog = pd.concat([catalog.reset_index(drop=True), vals_df], axis=1)
        evaluated = catalog["value_numeric"].notna().sum()
        print(f"  Evaluated: {evaluated}/{len(catalog)} returned numeric values ✓")

    MODEL_CATALOG = catalog
    return catalog


# Globals populated by explore_model
MODEL_CATALOG = None
TMSL_OBJ = None


In [ ]:
catalog = explore_model(evaluate=True)
catalog[["measure_name", "table_name", "expression", "value_numeric"]].head(20)


---

## Step 4 — What-If: Modify a measure safely

This is the core of Playground. Pick any measure, write a modified DAX expression, and see the original vs. modified values side-by-side. **The live model is never touched** — modified DAX is evaluated via `evaluate_dax()` as an inline query.


In [ ]:
def whatif_measure(measure_name: str, modified_dax: str, groupby_columns: list = None):
    """Evaluate a measure with modified DAX, side-by-side with the original.

    The original value comes from ``evaluate_measure()`` (uses the live model
    definition). The modified value comes from ``evaluate_dax()`` with your
    new DAX expression injected as an inline ``DEFINE MEASURE`` override —
    **no writes to the model**.

    Args:
        measure_name:    Name of the measure to experiment with.
        modified_dax:    Your alternative DAX expression (e.g.
                         ``"SUM(fact_sale[TotalIncludingTax]) * 0.9"``).
        groupby_columns: Optional list of ``"'Table'[Column]"`` strings to
                         break the comparison down by dimension.

    Returns:
        dict with keys:
            - ``"original_value"``: current model value (numeric)
            - ``"modified_value"``: sandbox value (numeric)
            - ``"delta"``: absolute change
            - ``"pct_change"``: relative change as fraction
            - ``"comparison_df"``: DataFrame with full comparison
            - ``"measure_name"``, ``"original_dax"``, ``"modified_dax"``

    Example:
        >>> whatif_measure("Total Sales",
        ...               "SUM(fact_sale[TotalIncludingTax]) * 0.9")
    """
    if MODEL_CATALOG is None:
        raise RuntimeError("Run explore_model() first.")

    # Look up the original DAX
    row = MODEL_CATALOG[MODEL_CATALOG["measure_name"] == measure_name]
    if row.empty:
        raise ValueError(f"Measure '{measure_name}' not found. "
                         f"Available: {MODEL_CATALOG['measure_name'].tolist()}")
    original_dax = row.iloc[0]["expression"]
    table_name = row.iloc[0]["table_name"]

    print(f"What-If: '{measure_name}'")
    print(f"  Original DAX: {original_dax}")
    print(f"  Modified DAX: {modified_dax}")

    # --- Evaluate original ---
    orig_num, orig_str = _evaluate_single_measure(measure_name)
    print(f"  Original value: {orig_num}")

    # --- Evaluate modified via DEFINE MEASURE override ---
    if groupby_columns:
        cols = ", ".join(groupby_columns)
        dax_query = (
            f"DEFINE MEASURE '{table_name}'[__playground_whatif] = {modified_dax}\n"
            f"EVALUATE SUMMARIZECOLUMNS({cols}, "
            f"\"__original\", [{measure_name}], "
            f"\"__modified\", '{table_name}'[__playground_whatif])"
        )
    else:
        dax_query = (
            f"DEFINE MEASURE '{table_name}'[__playground_whatif] = {modified_dax}\n"
            f"EVALUATE ROW(\"__original\", [{measure_name}], "
            f"\"__modified\", '{table_name}'[__playground_whatif])"
        )

    try:
        result_df = fabric.evaluate_dax(
            dataset=DATASET_NAME,
            workspace=WORKSPACE_NAME,
            dax_string=dax_query,
        )
    except Exception as e:
        print(f"  ✗ DAX evaluation failed: {e}")
        print(f"  Query was:\n{dax_query}")
        return {"error": str(e), "dax_query": dax_query}

    if groupby_columns:
        # Multi-row result
        result_df["__delta"] = result_df["[__modified]"] - result_df["[__original]"]
        result_df["__pct_change"] = result_df.apply(
            lambda r: r["__delta"] / r["[__original]"]
            if r["[__original]"] and r["[__original]"] != 0 else None, axis=1)
        print(f"  Modified values evaluated across {len(result_df)} groups ✓")

        from IPython.display import display, Markdown
        display(Markdown(f"### What-If: `{measure_name}`\n"
                         f"**Original DAX:** `{original_dax}`\n\n"
                         f"**Modified DAX:** `{modified_dax}`"))
        display(result_df)
        _plot_whatif_grouped(result_df, measure_name, groupby_columns)

        return {
            "measure_name": measure_name,
            "original_dax": original_dax,
            "modified_dax": modified_dax,
            "comparison_df": result_df,
        }
    else:
        # Single-row result
        mod_val = None
        try:
            mod_val = float(result_df.iloc[0]["[__modified]"])
        except (TypeError, ValueError, IndexError):
            pass
        orig_val = None
        try:
            orig_val = float(result_df.iloc[0]["[__original]"])
        except (TypeError, ValueError, IndexError):
            orig_val = orig_num

        delta = (mod_val - orig_val) if (mod_val is not None and orig_val is not None) else None
        pct = (delta / orig_val) if (delta is not None and orig_val and orig_val != 0) else None

        print(f"  Modified value: {mod_val}")
        if delta is not None:
            direction = "▲" if delta > 0 else "▼" if delta < 0 else "="
            pct_str = f"{pct:.2%}" if pct is not None else "n/a"
            print(f"  Delta: {direction} {abs(delta):,.2f} ({pct_str})")

        from IPython.display import display, Markdown
        display(Markdown(
            f"### What-If: `{measure_name}`\n\n"
            f"| | Value |\n|---|---|\n"
            f"| **Original** (`{original_dax}`) | {orig_val:,.2f} |\n"
            f"| **Modified** (`{modified_dax}`) | {mod_val:,.2f} |\n"
            f"| **Delta** | {direction} {abs(delta):,.2f} ({pct_str}) |"
        ))
        _plot_whatif_single(orig_val, mod_val, measure_name)

        return {
            "measure_name":   measure_name,
            "original_dax":   original_dax,
            "modified_dax":   modified_dax,
            "original_value": orig_val,
            "modified_value": mod_val,
            "delta":          delta,
            "pct_change":     pct,
            "comparison_df":  result_df,
        }


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def _plot_whatif_single(orig_val, mod_val, measure_name):
    """Bar chart comparing original vs modified value for a single measure."""
    if orig_val is None or mod_val is None:
        return
    fig, ax = plt.subplots(figsize=(6, 3.5))
    bars = ax.bar(["Original", "Modified"], [orig_val, mod_val],
                  color=["#4472C4", "#ED7D31"], width=0.5, edgecolor="white")
    # Annotate values on bars
    for bar, val in zip(bars, [orig_val, mod_val]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val:,.0f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_title(f"What-If: {measure_name}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Value")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


def _plot_whatif_grouped(df, measure_name, groupby_columns):
    """Grouped bar chart for dimensional what-if comparison."""
    if df.empty:
        return
    # Use first groupby column as label
    label_col = [c for c in df.columns if c not in
                 ["[__original]", "[__modified]", "__delta", "__pct_change"]][0]
    plot_df = df.sort_values("[__original]", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.4)))
    y = range(len(plot_df))
    h = 0.35
    ax.barh([i + h/2 for i in y], plot_df["[__original]"], h,
            label="Original", color="#4472C4", edgecolor="white")
    ax.barh([i - h/2 for i in y], plot_df["[__modified]"], h,
            label="Modified", color="#ED7D31", edgecolor="white")
    ax.set_yticks(list(y))
    ax.set_yticklabels(plot_df[label_col].astype(str).tolist(), fontsize=9)
    ax.set_xlabel("Value")
    ax.set_title(f"What-If: {measure_name} (by {label_col})", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


---

## Step 5 — Impact Analysis: What else changes?

When you modify a measure, other measures that **reference** it in their DAX will produce different results too. `impact_analysis()` finds every downstream dependent measure, re-evaluates each one with your modified DAX injected, and shows the full cascade of changes.


In [ ]:
def _find_dependent_measures(target_measure_name, catalog):
    """Find measures whose DAX references the target measure.

    Uses bracket-reference pattern matching: any measure whose expression
    contains ``[Target Measure Name]`` is considered a dependent.

    Args:
        target_measure_name: The measure being modified.
        catalog: The MODEL_CATALOG DataFrame.

    Returns:
        List of (measure_name, table_name, expression) tuples for dependents.
    """
    dependents = []
    pattern = re.compile(re.escape(f"[{target_measure_name}]"), re.IGNORECASE)
    for _, row in catalog.iterrows():
        if row["measure_name"] == target_measure_name:
            continue
        expr = row.get("expression") or ""
        if pattern.search(expr):
            dependents.append((row["measure_name"], row["table_name"], expr))
    return dependents


def impact_analysis(measure_name: str, modified_dax: str):
    """Evaluate downstream impact of modifying a measure.

    Finds every measure that references ``measure_name`` in its DAX,
    then evaluates each one twice: once with the original model definition,
    and once with your modified DAX injected via ``DEFINE MEASURE``. Shows
    the full cascade of value changes.

    Args:
        measure_name: The measure you're modifying.
        modified_dax: Your alternative DAX expression.

    Returns:
        pandas.DataFrame with columns: dependent_measure, original_value,
        modified_value, delta, pct_change.

    Example:
        >>> impact_analysis("Total Sales",
        ...                 "SUM(fact_sale[TotalIncludingTax]) * 0.9")
    """
    if MODEL_CATALOG is None:
        raise RuntimeError("Run explore_model() first.")

    row = MODEL_CATALOG[MODEL_CATALOG["measure_name"] == measure_name]
    if row.empty:
        raise ValueError(f"Measure '{measure_name}' not found.")
    table_name = row.iloc[0]["table_name"]

    dependents = _find_dependent_measures(measure_name, MODEL_CATALOG)
    print(f"Impact analysis for '{measure_name}'")
    print(f"  Found {len(dependents)} dependent measure(s): "
          f"{[d[0] for d in dependents]}")

    if not dependents:
        print("  No downstream measures reference this one — impact is isolated.")
        from IPython.display import display, Markdown
        display(Markdown(f"### Impact Analysis: `{measure_name}`\n\n"
                         f"No other measures reference `[{measure_name}]` in their DAX. "
                         f"This change is **isolated** — only this measure is affected."))
        return pd.DataFrame()

    impact_rows = []
    for dep_name, dep_table, dep_expr in dependents:
        # Original value
        orig_num, _ = _evaluate_single_measure(dep_name)

        # Modified value: inject the modified measure definition
        dax_query = (
            f"DEFINE MEASURE '{table_name}'[{measure_name}] = {modified_dax}\n"
            f"EVALUATE ROW(\"__value\", [{dep_name}])"
        )
        mod_num = None
        try:
            result = fabric.evaluate_dax(
                dataset=DATASET_NAME,
                workspace=WORKSPACE_NAME,
                dax_string=dax_query,
            )
            mod_num = float(result.iloc[0, 0]) if len(result) else None
        except Exception as e:
            print(f"  [warn] Could not evaluate '{dep_name}' with override: {e}")

        delta = (mod_num - orig_num) if (mod_num is not None and orig_num is not None) else None
        pct = (delta / orig_num) if (delta is not None and orig_num and orig_num != 0) else None

        impact_rows.append({
            "dependent_measure": dep_name,
            "original_value":    orig_num,
            "modified_value":    mod_num,
            "delta":             delta,
            "pct_change":        pct,
        })
        direction = "▲" if delta and delta > 0 else "▼" if delta and delta < 0 else "="
        print(f"  {dep_name}: {orig_num} → {mod_num} ({direction})")

    impact_df = pd.DataFrame(impact_rows)

    # Render
    from IPython.display import display, Markdown
    lines = [f"### Impact Analysis: `{measure_name}`\n",
             f"Modified DAX: `{modified_dax}`\n",
             f"| Dependent Measure | Original | Modified | Delta | Change |",
             f"|---|---|---|---|---|"]
    for _, r in impact_df.iterrows():
        o = f"{r['original_value']:,.2f}" if r['original_value'] is not None else "n/a"
        m = f"{r['modified_value']:,.2f}" if r['modified_value'] is not None else "n/a"
        d = f"{r['delta']:,.2f}" if r['delta'] is not None else "n/a"
        p = f"{r['pct_change']:.2%}" if r['pct_change'] is not None else "n/a"
        lines.append(f"| {r['dependent_measure']} | {o} | {m} | {d} | {p} |")
    display(Markdown("\n".join(lines)))

    _plot_impact(impact_df, measure_name)
    return impact_df


def _plot_impact(impact_df, source_measure):
    """Horizontal bar chart showing delta for each impacted measure."""
    df = impact_df.dropna(subset=["delta"])
    if df.empty:
        return
    df = df.sort_values("delta", key=abs, ascending=True)

    fig, ax = plt.subplots(figsize=(8, max(3, len(df) * 0.6)))
    colors = ["#C0392B" if d < 0 else "#27AE60" for d in df["delta"]]
    ax.barh(df["dependent_measure"], df["delta"], color=colors, edgecolor="white")
    for i, (_, r) in enumerate(df.iterrows()):
        pct = f"{r['pct_change']:.1%}" if r['pct_change'] is not None else ""
        ax.text(r['delta'], i, f"  {pct}", va="center", fontsize=9,
                color="#333", fontweight="bold")
    ax.set_xlabel("Value Change (Delta)")
    ax.set_title(f"Downstream Impact of Modifying '{source_measure}'",
                 fontsize=12, fontweight="bold")
    ax.axvline(0, color="#888", linewidth=0.8, linestyle="--")
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


---

## Step 6 — Compare multiple scenarios

Sometimes you want to test several alternative formulas for the same measure. `compare_scenarios()` evaluates each one and shows them all in a single comparison table and chart.


In [ ]:
def compare_scenarios(measure_name: str, scenarios: dict):
    """Evaluate multiple alternative DAX expressions for a single measure.

    Args:
        measure_name: The measure to experiment with.
        scenarios:    Dict mapping scenario label → modified DAX expression.
                      Example: {
                          "Exclude returns": "SUM(fact_sale[TotalIncludingTax]) - SUM(fact_sale[Returns])",
                          "10% discount":    "SUM(fact_sale[TotalIncludingTax]) * 0.9",
                          "Round to 1000s":  "ROUND(SUM(fact_sale[TotalIncludingTax]), -3)",
                      }

    Returns:
        pandas.DataFrame with one row per scenario + the original.

    Example:
        >>> compare_scenarios("Total Sales", {
        ...     "Exclude returns": "SUM(...) - SUM(...)",
        ...     "10% discount": "SUM(...) * 0.9",
        ... })
    """
    if MODEL_CATALOG is None:
        raise RuntimeError("Run explore_model() first.")

    row = MODEL_CATALOG[MODEL_CATALOG["measure_name"] == measure_name]
    if row.empty:
        raise ValueError(f"Measure '{measure_name}' not found.")
    table_name = row.iloc[0]["table_name"]
    original_dax = row.iloc[0]["expression"]

    # Evaluate original
    orig_num, _ = _evaluate_single_measure(measure_name)

    results = [{"scenario": "Original (current)",
                "dax": original_dax,
                "value": orig_num, "delta": 0, "pct_change": 0}]

    for label, dax_expr in scenarios.items():
        dax_query = (
            f"DEFINE MEASURE '{table_name}'[__playground_whatif] = {dax_expr}\n"
            f"EVALUATE ROW(\"__value\", '{table_name}'[__playground_whatif])"
        )
        mod_val = None
        try:
            result = fabric.evaluate_dax(
                dataset=DATASET_NAME, workspace=WORKSPACE_NAME,
                dax_string=dax_query)
            mod_val = float(result.iloc[0, 0]) if len(result) else None
        except Exception as e:
            print(f"  [warn] Scenario '{label}' failed: {e}")

        delta = (mod_val - orig_num) if (mod_val is not None and orig_num is not None) else None
        pct = (delta / orig_num) if (delta is not None and orig_num and orig_num != 0) else None
        results.append({"scenario": label, "dax": dax_expr,
                        "value": mod_val, "delta": delta, "pct_change": pct})

    results_df = pd.DataFrame(results)

    # Render
    from IPython.display import display, Markdown
    display(Markdown(f"### Scenario Comparison: `{measure_name}`"))
    display(results_df[["scenario", "value", "delta", "pct_change"]])

    # Chart
    fig, ax = plt.subplots(figsize=(8, 3.5))
    colors = ["#4472C4"] + ["#ED7D31"] * len(scenarios)
    ax.bar(results_df["scenario"], results_df["value"], color=colors,
           edgecolor="white", width=0.6)
    for i, (_, r) in enumerate(results_df.iterrows()):
        if r["value"] is not None:
            ax.text(i, r["value"], f"{r['value']:,.0f}", ha="center",
                    va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(f"Scenario Comparison: {measure_name}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Value")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()

    return results_df


---

## Demo Walkthrough

The following cells demonstrate Playground against the configured semantic model. Each cell is a self-contained experiment you can run, modify, and re-run.

### Demo 1: Simple what-if — "What if Total Sales dropped 10%?"


In [ ]:
# Simple what-if: multiply Total Sales by 0.9
result = whatif_measure(
    measure_name = "Total Sales",
    modified_dax = "SUM(fact_sale[TotalIncludingTax]) * 0.9",
)


### Demo 2: Downstream impact — "If I change Total Sales, what else moves?"


In [ ]:
# Impact analysis: which other measures reference Total Sales?
impact_df = impact_analysis(
    measure_name = "Total Sales",
    modified_dax = "SUM(fact_sale[TotalIncludingTax]) * 0.9",
)


### Demo 3: Compare multiple formula options


In [ ]:
# Compare three different approaches to calculating Total Sales
compare_scenarios("Total Sales", {
    "Exclude tax":   "SUM(fact_sale[TotalExcludingTax])",
    "10% discount":  "SUM(fact_sale[TotalIncludingTax]) * 0.9",
    "Round to 1000": "ROUND(SUM(fact_sale[TotalIncludingTax]), -3)",
})


### Demo 4: What-if with dimensional breakdown

You can also break down the comparison by a dimension to see how the change plays out across categories.


In [ ]:
# What-if broken down by a dimension
# NOTE: adjust the groupby column to match your model's actual column names
result = whatif_measure(
    measure_name   = "Total Sales",
    modified_dax   = "SUM(fact_sale[TotalIncludingTax]) * 0.9",
    groupby_columns = ["'dimension_city'[StateProvince]"],
)


---

## How to adopt Playground in YOUR workspace

1. **Clone this notebook** into any Fabric workspace.
2. **Edit the config cell** at the top — set `WORKSPACE_NAME` and `DATASET_NAME`.
3. **Run Step 3** to load your model's measure catalog.
4. **Experiment freely** — use `whatif_measure()`, `impact_analysis()`, and `compare_scenarios()` with any measure and any DAX modification.

### Suggested workflows

- **Pre-edit validation:** Before changing a measure in the live model, test the new DAX in Playground first. Confirm the value change is what you expect AND that no downstream KPIs are unexpectedly affected.
- **Formula comparison:** When debating between two approaches to a calculation, run both through `compare_scenarios()` and share the chart with your team.
- **Onboarding:** New team members can use Playground to understand what each measure does by experimenting with the DAX without any risk.
- **Sprint planning:** Before a model refactor sprint, run `impact_analysis()` on each planned change to estimate blast radius.

---

## Known limitations

- **Values are total-level by default:** `whatif_measure()` evaluates at the grand total unless you pass `groupby_columns`. This is fast but doesn't catch filter-context-dependent bugs. Extend with specific dimensional breakdowns for deeper testing.
- **Dependency detection is regex-based:** `_find_dependent_measures` matches `[Measure Name]` in DAX text. This catches direct references but may miss measures invoked via `CALCULATE` with table-scoped syntax or via calculation groups.
- **No row-level data mutation:** Playground modifies measure *formulas*, not underlying data. "What if this table had 10% more rows?" is out of scope — that's a different tool.
- **DEFINE MEASURE scope:** The `DEFINE MEASURE` override in DAX queries is scoped to a single query. Deeply nested chains (A → B → C → D) are evaluated correctly for the direct override but intermediate measures use the live definitions unless explicitly overridden too.

## Extension ideas

- **Persistent scenarios:** Save what-if results to a Delta table, track how scenarios evolve over time.
- **Approval workflow:** Require Playground validation before any measure edit in production — integrate with Data Pipelines.
- **Multi-measure override:** Extend `DEFINE MEASURE` to override multiple measures in a single query for complex refactoring scenarios.
- **Visual diff:** Side-by-side chart showing value distributions across all dimensions, not just totals.

---

## Credits & citations

- **semantic-link** — https://github.com/microsoft/semantic-link (`sempy`)
- **semantic-link-labs** — https://github.com/microsoft/semantic-link-labs (`sempy_labs`)
- **Wide World Importers sample** — Microsoft Fabric built-in sample dataset

---

*Submitted to the Fabric Semantic Link Developer Experience Challenge, April 2026.*
